In [1]:
# =========================================================
# KAGGLE TOP 1% GRANDMASTER PIPELINE - FIXED VERSION
# =========================================================

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

DATA="/kaggle/input/competitions/march-machine-learning-mania-2026/"

# =========================
# LOAD DATA
# =========================

M_reg = pd.read_csv(DATA+"MRegularSeasonDetailedResults.csv")
W_reg = pd.read_csv(DATA+"WRegularSeasonDetailedResults.csv")

M_comp = pd.read_csv(DATA+"MRegularSeasonCompactResults.csv")
W_comp = pd.read_csv(DATA+"WRegularSeasonCompactResults.csv")

M_tour = pd.read_csv(DATA+"MNCAATourneyCompactResults.csv")
W_tour = pd.read_csv(DATA+"WNCAATourneyCompactResults.csv")

M_seeds = pd.read_csv(DATA+"MNCAATourneySeeds.csv")
W_seeds = pd.read_csv(DATA+"WNCAATourneySeeds.csv")

massey = pd.read_csv(DATA+"MMasseyOrdinals.csv")

sub = pd.read_csv(DATA+"SampleSubmissionStage1.csv")

# =========================
# COMBINE MEN + WOMEN
# =========================

reg = pd.concat([M_reg,W_reg],ignore_index=True)
compact = pd.concat([M_comp,W_comp],ignore_index=True)
tour = pd.concat([M_tour,W_tour],ignore_index=True)
seeds = pd.concat([M_seeds,W_seeds],ignore_index=True)

# =========================
# SEED FEATURE
# =========================

seeds["SeedNum"] = seeds["Seed"].str[1:3].astype(int)
seeds = seeds[["Season","TeamID","SeedNum"]]

# =========================
# WIN RATE
# =========================

wins = compact.groupby(["Season","WTeamID"]).size().reset_index(name="Wins")
loss = compact.groupby(["Season","LTeamID"]).size().reset_index(name="Loss")

stats = wins.merge(loss,
                   left_on=["Season","WTeamID"],
                   right_on=["Season","LTeamID"],
                   how="outer").fillna(0)

stats["TeamID"] = stats["WTeamID"].fillna(stats["LTeamID"]).astype(int)
stats["Games"] = stats["Wins"] + stats["Loss"]
stats["WinRate"] = stats["Wins"] / stats["Games"]

stats = stats[["Season","TeamID","WinRate","Games"]]

# =========================
# SCORE DIFFERENCE
# =========================

compact["ScoreDiff"] = compact["WScore"] - compact["LScore"]

score_w = compact.groupby(["Season","WTeamID"])["ScoreDiff"].mean().reset_index()
score_l = compact.groupby(["Season","LTeamID"])["ScoreDiff"].mean().reset_index()

score_w.columns=["Season","TeamID","ScoreW"]
score_l.columns=["Season","TeamID","ScoreL"]

score = score_w.merge(score_l,on=["Season","TeamID"],how="outer").fillna(0)

score["ScoreStrength"] = score["ScoreW"] - score["ScoreL"]
score = score[["Season","TeamID","ScoreStrength"]]

# =========================
# MASSEY RANK
# =========================

massey = massey[massey["RankingDayNum"]==133]

rank = massey.groupby(["Season","TeamID"])["OrdinalRank"].mean().reset_index()
rank.columns=["Season","TeamID","MasseyRank"]

# =========================
# TEAM FEATURE TABLE
# =========================

team = stats.merge(score,on=["Season","TeamID"],how="left")
team = team.merge(seeds,on=["Season","TeamID"],how="left")
team = team.merge(rank,on=["Season","TeamID"],how="left")

# =========================
# BUILD TRAIN
# =========================

rows=[]

for _,r in tour.iterrows():

    rows.append([r.Season,r.WTeamID,r.LTeamID,1])
    rows.append([r.Season,r.LTeamID,r.WTeamID,0])

train=pd.DataFrame(rows,columns=["Season","Team1","Team2","Target"])

# =========================
# ATTACH FEATURES
# =========================

def attach(df):

    t1 = team.copy()
    t2 = team.copy()

    t1.columns = ["Season","Team1","T1_WR","T1_Games","T1_Score","T1_Seed","T1_Rank"]
    t2.columns = ["Season","Team2","T2_WR","T2_Games","T2_Score","T2_Seed","T2_Rank"]

    df = df.merge(t1,on=["Season","Team1"],how="left")
    df = df.merge(t2,on=["Season","Team2"],how="left")

    df["WR_diff"] = df["T1_WR"] - df["T2_WR"]
    df["Score_diff"] = df["T1_Score"] - df["T2_Score"]
    df["Seed_diff"] = df["T2_Seed"] - df["T1_Seed"]
    df["Rank_diff"] = df["T2_Rank"] - df["T1_Rank"]

    return df

train = attach(train)

features = [
"WR_diff",
"Score_diff",
"Seed_diff",
"Rank_diff"
]

X = train[features].fillna(0)
y = train["Target"]

# =========================
# MODELS
# =========================

xgb = XGBClassifier(
n_estimators=1500,
learning_rate=0.01,
max_depth=4,
subsample=0.8,
colsample_bytree=0.8,
eval_metric="logloss",
tree_method="hist"
)

lgb = LGBMClassifier(
n_estimators=1500,
learning_rate=0.01,
subsample=0.8,
colsample_bytree=0.8
)

xgb.fit(X,y)
lgb.fit(X,y)

# =========================
# PREDICTION
# =========================

sub[["Season","Team1","Team2"]] = sub["ID"].str.split("_",expand=True).astype(int)

sub = attach(sub)

pred1 = xgb.predict_proba(sub[features].fillna(0))[:,1]
pred2 = lgb.predict_proba(sub[features].fillna(0))[:,1]

sub["Pred"] = (pred1 + pred2)/2

sub[["ID","Pred"]].to_csv("submission.csv",index=False)

print("Submission created successfully!")

[LightGBM] [Info] Number of positive: 4302, number of negative: 4302
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 794
[LightGBM] [Info] Number of data points in the train set: 8604, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Submission created successfully!
